In [0]:
import os
import json

volume_path = "/Volumes/sre/dev/friday"

# Get processed files from Delta table
processed_files = spark.sql("SELECT DISTINCT file_name FROM `sre`.dev.slack_docs_track").collect()
processed_files = set(row["file_name"] for row in processed_files)

# Process new files
new_files = [f for f in os.listdir(volume_path) if f not in processed_files]

new_files

In [0]:
%sql
SELECT ai_summarize(
    'Apache Spark is a unified analytics engine for large-scale data processing. ' ||
    'It provides high-level APIs in Java, Scala, Python and R, and an optimized ' ||
    'engine that supports general execution graphs. It also supports a rich set ' ||
    'of higher-level tools including Spark SQL for SQL and structured data ' ||
    'processing, pandas API on Spark for pandas workloads, MLlib for machine ' ||
    'learning, GraphX for graph processing, and Structured Streaming for incremental ' ||
    'computation and stream processing.', 500
) AS spark_summary;


In [0]:
file_path = "/Volumes/sre/dev/friday/test.json"
with open(file_path, 'r') as file:
    data = json.load(file)
display(data)

In [0]:
messages = data["messages"]

In [0]:
display(messages)

In [0]:
from pyspark.sql import functions as F

# Create DataFrame with raw message data
df = spark.createDataFrame(
    [
        {
            "has_thread": msg["has_thread"],
            "thread_replies": msg.get("thread_replies", []),
            "text": msg["text"]
        } 
        for msg in messages
    ]
)

# Process text column
df = df.withColumn(
    "text",
    F.when(
        F.col("has_thread"),
        # Aggregate all thread replies' text and summarize
        F.expr("ai_summarize(aggregate(thread_replies, '', (acc, x) -> concat(acc, ' ', x.text)), 500)")
    ).otherwise(
        F.col("text")  # Original text if no thread
    )
)


In [0]:
display(df)

First Major Test

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

# Current date for timestamp comparison
current_date = datetime.now()

# Convert messages array to DataFrame
messages_df = spark.createDataFrame(messages)

# Create a window specification ordered by timestamp
window_spec = Window.orderBy("timestamp")

In [0]:
# Mark group boundaries - new group starts when:
# 1. A message has has_thread=true
# 2. A message timestamp exceeds current date
# 3. Previous message had has_thread=true
messages_df = messages_df.withColumn(
    "prev_has_thread", 
    F.lag("has_thread", 1, False).over(window_spec)
)

In [0]:
messages_df = messages_df.withColumn(
    "date",
    F.to_date("timestamp")
)

messages_df = messages_df.withColumn(
    "prev_date",
    F.lag("date", 1).over(window_spec)
)

In [0]:
# messages_df = messages_df.withColumn(
#     "new_group",
#     (F.col("has_thread") == True) | 
#     (F.col("timestamp") > F.lit(current_date)) |
#     (F.row_number().over(window_spec) == 1) |
#     (F.col("prev_has_thread") == True)
# )

messages_df = messages_df.withColumn(
    "new_group",
    (F.col("has_thread") == True) | 
    (F.col("timestamp") > F.lit(current_date)) |
    (F.row_number().over(window_spec) == 1) |
    (F.col("prev_has_thread") == True) |
    (F.col("date") != F.col("prev_date"))   # <-- NEW CONDITION
)


In [0]:
# Create group IDs using cumulative sum of the new_group column
messages_df = messages_df.withColumn(
    "group_id", 
    F.sum(F.when(F.col("new_group"), 1).otherwise(0)).over(window_spec)
)

In [0]:
# Format message text for display: "username: text"
messages_df = messages_df.withColumn(
    "formatted_text",
    F.concat(F.col("username"), F.lit(": "), F.col("text"))
)

In [0]:
display(messages_df)

In [0]:
# Process non-threaded messages - group and collect
non_thread_df = messages_df.filter(F.col("has_thread") == False)
non_thread_result = non_thread_df.groupBy("group_id").agg(
    F.collect_list("formatted_text").alias("text"),
    F.min("timestamp").alias("timestamp")
)

In [0]:
display(non_thread_result)

In [0]:
from pyspark.sql import functions as F

# Process threaded messages to create formatted reply arrays
thread_result = (
    messages_df.filter(F.col("has_thread"))
    .select(
        "group_id",
        "thread_replies",
        F.col("timestamp").alias("msg_timestamp")
    )
    .withColumn(
        # Transform each reply to "username: text" format
        "text",
        F.transform(
            "thread_replies",
            lambda r: F.concat(r["username"], F.lit(": "), r["text"])
        )
    )
    .withColumn(
        # Get minimum timestamp from replies or use message timestamp
        "min_timestamp",
        F.coalesce(
            F.array_min(F.transform("thread_replies", lambda r: r["timestamp"])),
            F.col("msg_timestamp")
        )
    )
    .select("group_id", "text", F.col("min_timestamp").alias("timestamp"))
)

In [0]:
display(thread_result)

In [0]:
# Combine both results
final_df = non_thread_result.union(thread_result)

# Order by timestamp and select only needed columns
result_df = final_df.orderBy("timestamp").select("text", "timestamp")

# Display results
display(result_df)

In [0]:
from pyspark.sql import functions as F

# Convert timestamp to date and summarize text
result_df = (result_df
    .withColumn("date", F.to_date("timestamp"))  # Convert to yyyy-MM-dd format
    .withColumn("combined_text", F.concat_ws(" ", "text"))  # Flatten text array
    .withColumn("text", F.expr("ai_summarize(combined_text, 500)"))  # Summarize text
    .select("date", "text")  # Keep final columns
)

In [0]:
display(result_df)

In [0]:
%sql
SELECT * FROM `sre`.dev.slack_docs_index

In [0]:
%sql
SELECT * FROM `sre`.dev.slack_docs_text

In [0]:
%sql
CREATE OR REPLACE FUNCTION `sre`.dev.slack_docs_vector_search (
  -- The agent uses this comment to determine how to generate the query string parameter.
  query STRING
  COMMENT 'The query string for searching Slack chat.'
) RETURNS TABLE
-- The agent uses this comment to determine when to call this tool. It describes the types of documents and information contained within the index.
COMMENT 'Executes a search on slack chat summaries to retrieve text documents most relevant to the input query.' RETURN
SELECT
  text as page_content,
  map('date', date, 'id', id) as metadata
FROM
  vector_search(
    -- Specify your Vector Search index name here
    index => 'sre.dev.slack_docs_index',
    query => query,
    num_results => 5
  )

In [0]:
%sql
DROP FUNCTION `sre`.dev.slack_docs_by_date

In [0]:
%sql
CREATE OR REPLACE FUNCTION `sre`.dev.slack_docs_by_date (
  query_date STRING
  COMMENT 'The date to filter Slack messages by (format: YYYY-MM-DD)'
) RETURNS TABLE
COMMENT 'Retrieves Slack messages from a specific date.' 
RETURN
SELECT
  text as page_content,
  map('date', date, 'id', id) as metadata
FROM
  `sre`.dev.slack_docs_text
WHERE
  date = query_date
